## Homework Reflection: Weeks 5–8 Code Compilation
### Victor Vasquez Gil

Faculty of Computing and Data Science, Boston University
DX702 Experimental Design and Causality

The written reflections and causal diagrams are in the separate PDF. This notebook contains only the code I used for the questions that require it. For Week 8, I load the quiz datasets directly from the instructor's public GitHub repository.

I imported the packages for the simulations, regressions, and matching. For Week 8, the read_quiz function reads the CSV files directly from the course repository. This way, the code does not depend on where I saved the notebook in Codespaces.

In [17]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import cdist

def read_quiz(stem):
    url = (
        "https://raw.githubusercontent.com/"
        "joshua-vonkorff/DX702-mod-6/main/"
        f"{stem}.csv"
    )
    return pd.read_csv(url)

### Week 5 - Question 3: Wildlife simulation
I simulated 500 observations to follow the relationships in the question. Storms happen randomly, and I added separate noise to bears, deer, and flowers so the numbers do not follow a perfect pattern. The other Week 5 questions use diagrams and written explanations in the PDF.

In [18]:
rng = np.random.default_rng(2026)
storms = rng.poisson(2, 500)
bears = np.maximum(0, 60 - 7*storms + rng.normal(0, 5, 500))
deer = np.maximum(0, 120 - 9*storms - .35*bears + rng.normal(0, 7, 500))
flowers = np.maximum(0, 180 + 12*storms - .45*deer + rng.normal(0, 9, 500))
wildlife = pd.DataFrame(dict(storms=storms, bears=bears, deer=deer, flowers=flowers))
print(wildlife.head().round(1))

   storms  bears  deer  flowers
0       1   46.9  99.2    150.9
1       2   44.8  84.3    178.3
2       2   47.5  83.6    170.2
3       4   28.3  72.0    202.0
4       4   33.9  78.8    208.3


### Week 6 - Question 2: Estimating an upper end treatment effect
I compared the largest observed effect with the 90th percentile across repeated samples. I wanted to see how much random noise can push the maximum upward and whether using a percentile gives a less extreme result. This is a proxy, not the exact maximum treatment effect.

In [19]:
rng = np.random.default_rng(702)
true_effect = np.linspace(.5, 2, 1000)
estimates = [true_effect + rng.normal(0, .7, 1000) for _ in range(600)]
print('True maximum:', true_effect.max())
print('Average observed maximum:', round(np.mean([e.max() for e in estimates]), 3))
print('Average 90th percentile:', round(np.mean([np.percentile(e, 90) for e in estimates]), 3))

True maximum: 2.0
Average observed maximum: 3.852
Average 90th percentile: 2.308


### Week 7 - Omitted-variable bias and repeated testing
For Question 1, I gave W a positive effect on both X and Y. I ran the regression with W left out and then with W included to see how the coefficient of X changed.

For Question 2, I made W independent of the outcome and ran the regression 1,000 times. I checked the smallest p-value and counted how often W looked significant, even though its true coefficient was zero.

In [20]:
# Question 1 — Leaving out a confounder
rng = np.random.default_rng(702)
w = rng.normal(size=5000)
x = .8*w + rng.normal(size=5000)
y = 2*x + 1.5*w + rng.normal(size=5000)
without_w = sm.OLS(y, sm.add_constant(x)).fit().params[1]
with_w = sm.OLS(y, sm.add_constant(np.column_stack([x, w]))).fit().params[1]
print('True = 2; omitted W =', round(without_w, 3), '; with W =', round(with_w, 3))

True = 2; omitted W = 2.724 ; with W = 1.987


In [22]:
# Question 2 - Repeating the regression 1,000 times
rng = np.random.default_rng(2026)
pvals = []
for _ in range(1000):
    w, x = rng.normal(size=(2, 300))
    y = 2*x + rng.normal(size=300)
    pvals.append(sm.OLS(y, sm.add_constant(np.column_stack([x, w]))).fit().pvalues[2])
print('Smallest p-value:', round(min(pvals), 6))
print('False positives at 0.05:', sum(p < .05 for p in pvals))

Smallest p-value: 0.000241
False positives at 0.05: 46


### Week 8 — Coding quiz problems

For Quiz 8.1, I estimated treatment probabilities from Z and used normalized inverse-probability weights to compare treated and untreated outcomes. The main challenge was making sure the weights were applied to the correct group.

For Quiz 8.2, I matched each treated observation to its nearest control using Z1 and Z2. I used Mahalanobis distance to account for the variables' scales and correlation. Both datasets are loaded directly from the instructor's GitHub repository, so I do not need to upload CSV files separately.

In [ ]:
# Quiz 8.1 - Normalized inverse-probability weighting
d = read_quiz('homework_8.1')
p = LogisticRegression(max_iter=1000).fit(d[['Z']], d.X).predict_proba(d[['Z']])[:, 1]
t = d.X.to_numpy() == 1
y = d.Y.to_numpy()
ate = np.average(y[t], weights=1/p[t]) - np.average(y[~t], weights=1/(1-p[~t]))
print('Normalized IPW estimate:', round(ate, 5))

Normalized IPW estimate: 2.27434


In [24]:
#vQuiz 8.2 - Mahalanobis nearest-neighbor matching
d = read_quiz('homework_8.2')
t, c = d[d.X == 1], d[d.X == 0]
vi = np.linalg.inv(np.cov(d[['Z1', 'Z2']].to_numpy(), rowvar=False))
idx = cdist(t[['Z1', 'Z2']], c[['Z1', 'Z2']], metric='mahalanobis', VI=vi).argmin(axis=1)
print('Matched effect:', round((t.Y.to_numpy() - c.Y.to_numpy()[idx]).mean(), 5))

Matched effect: 3.43768
